# Titanic Survival Prediction
### James Koero — Kisumu, Kenya, 2026

I want to be upfront about something before starting. I'm redoing this notebook
because a professor told me my first version looked AI-generated. He was right
to question it. So this time I'm writing what I actually think, including the
parts where I'm unsure.

I studied physics at Moi University and worked at KenGen's Olkaria geothermal
site before getting into ML. I mention this because I tend to think about data
problems the way a physicist would — in terms of forces, interactions, and
whether the model's explanation of a system makes physical and social sense.

The Titanic dataset interests me because survival here wasn't purely random.
There were rules being enforced (women and children first), physical constraints
(time before the ship sank, lifeboat capacity), and social ones (who had access
to which deck). The ML question is: how much of that can we recover from the
12 columns we have?

My hypotheses going in, before I look at a single number:

Pclass should matter because the upper decks where lifeboats were loaded were
easier to reach from first class. Sex should matter because of the evacuation
protocol. Age might matter — children were prioritized — but the direction
for adults is less clear to me. Family size is my most uncertain hypothesis.
I think small families might do better than solo travelers, but I'm not sure
where the cutoff is.

Let me find out.

## Loading and first look

Before doing any analysis I want to actually read the first 10 rows and think
about what I'm looking at. I've made the mistake before of jumping straight to
value_counts() without really looking at the raw data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['figure.facecolor'] = '#0A1628'
matplotlib.rcParams['axes.facecolor'] = '#0A1628'
matplotlib.rcParams['axes.labelcolor'] = 'white'
matplotlib.rcParams['xtick.color'] = 'white'
matplotlib.rcParams['ytick.color'] = 'white'
matplotlib.rcParams['text.color'] = 'white'

GOLD = '#C9A84C'
BLUE = '#4A90D9'
RED  = '#E05555'

df = pd.read_csv('train.csv')
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print(f"Column names: {list(df.columns)}")
df.head(10)

The Name column immediately caught my attention. The names are formatted as
"Braund, Mr. Owen Harris" — there's a title embedded in there. Mr, Mrs, Miss,
Master. I hadn't thought about this before loading the data. Master specifically
means a young male child, which is different information from just knowing
someone is male. Miss means unmarried female, which correlates with being younger.

This could be more useful than the raw Age column, especially because Age has
a lot of missing values. I'll come back to this.

The Cabin column also looks very sparse just from glancing at these 10 rows.
I need to check how much is missing before deciding what to do with it.

## What's missing and does it matter?

I want to know not just how much is missing, but whether the missingness is
random or systematic. If the passengers with missing Age have a different
survival rate than those with known Age, I can't just drop those rows — I'd
be removing a non-random subset and biasing the training data.

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({'Count': missing, 'Percent': missing_pct})
print(missing_df[missing_df['Count'] > 0].sort_values('Percent', ascending=False))

Cabin is 77% missing. That's too much to use directly, but I can create a
binary flag — did this passenger have a recorded cabin or not. Passengers with
assigned cabins were likely wealthier and had better physical positioning on
the ship.

Age is about 20% missing, 177 passengers. That's significant enough that I
need a proper imputation strategy. Let me check the critical question first:
are the passengers with missing Age systematically different?

In [ ]:
missing_mask = df['Age'].isnull()
rate_known   = df[~missing_mask]['Survived'].mean()
rate_missing = df[missing_mask]['Survived'].mean()

print(f"Survival rate when Age is known:   {rate_known:.3f}")
print(f"Survival rate when Age is missing: {rate_missing:.3f}")
print()
print("Class breakdown for passengers with missing Age:")
print(df[missing_mask]['Pclass'].value_counts())

The survival rates are different. Passengers with missing Age survived at a
lower rate than those with known Age. And looking at the class breakdown,
most missing-Age passengers were in 3rd class. This confirms the missingness
is not random — it correlates with class and therefore with survival.

Dropping these rows would mean I'm training on a dataset that under-represents
3rd class passengers, which would give me an overoptimistic model. I need to
impute, not drop.

My plan: fill missing Age with the median age for each Pclass × Sex combination.
A 1st-class female passenger was probably older on average than a 3rd-class male.
Using a single overall median would be wrong for many rows.

## Exploring the survival patterns

I want to test my three main hypotheses — Pclass matters, Sex matters, and their
interaction. Rather than making 15 charts, I'll make three focused ones that
directly address what I'm trying to understand.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

pclass_surv = df.groupby('Pclass')['Survived'].mean()
axes[0].bar(pclass_surv.index, pclass_surv.values, color=GOLD, edgecolor='white', lw=0.5)
axes[0].set_title('By Passenger Class')
axes[0].set_xticks([1,2,3])
axes[0].set_xticklabels(['1st', '2nd', '3rd'])
axes[0].set_ylabel('Survival Rate')
for i, v in enumerate(pclass_surv.values):
    axes[0].text(i+1, v+0.01, f'{v:.2f}', ha='center', fontsize=11)

sex_surv = df.groupby('Sex')['Survived'].mean()
axes[1].bar(sex_surv.index, sex_surv.values, color=[BLUE, GOLD], edgecolor='white', lw=0.5)
axes[1].set_title('By Sex')
axes[1].set_ylabel('Survival Rate')
for i, (label, v) in enumerate(sex_surv.items()):
    axes[1].text(i, v+0.01, f'{v:.2f}', ha='center', fontsize=11)

cross = df.groupby(['Pclass','Sex'])['Survived'].mean().unstack()
x = np.arange(3)
w = 0.35
axes[2].bar(x-w/2, cross['female'], w, label='Female', color=GOLD, edgecolor='white')
axes[2].bar(x+w/2, cross['male'],   w, label='Male',   color=BLUE, edgecolor='white')
axes[2].set_title('Sex × Class Interaction')
axes[2].set_xticks(x)
axes[2].set_xticklabels(['1st','2nd','3rd'])
axes[2].legend(facecolor='#0A1628', labelcolor='white')

plt.tight_layout()
plt.savefig('survival_eda.png', dpi=150, bbox_inches='tight', facecolor='#0A1628')
plt.show()

The third chart is the one that really tells the story.

Second-class women survived at about 92%. That's higher than first-class men
at about 37%. Third-class women survived at roughly the same rate as first-class
men. What this means is that being female protected you more than being in first
class. The evacuation protocol overrode the class structure.

I had expected class to matter. I did not expect sex to so completely dominate
the class signal. In physics terms, think of it like two forces acting on the same
object. Class pushes survival probability upward. Sex creates an even stronger
force in the same direction for women and the opposite direction for men. The net
effect is dominated by the sex force.

This changes my thinking about which feature should come first in importance.
I'm going in expecting Sex to be the most powerful predictor.

## Building new features

Four things I want to create before training anything:

The title from the passenger's name. Family size as a single number (SibSp + Parch + 1).
A binary flag for traveling alone. And a binary flag for having a cabin.

Let me start with the title extraction because that one requires me to understand
the format of the Name column first.

In [ ]:
def get_title(name):
    # Format: "Surname, Title. Firstname Middlename"
    return name.split(',')[1].split('.')[0].strip()

# Test on a few names before applying to all 891 rows
for name in df['Name'].head(5):
    print(f"{name}  =>  '{get_title(name)}'")

The extraction works. Now let me see the full distribution of titles across
all passengers to decide how to handle the rare ones.

In [ ]:
df['Title'] = df['Name'].apply(get_title)
title_stats = df.groupby('Title')['Survived'].agg(['mean','count'])
title_stats.columns = ['SurvivalRate', 'Count']
print(title_stats.sort_values('Count', ascending=False))

There are 17 different titles. Mr (577), Miss (182), Mrs (125), Master (40) cover
almost everyone. Then there are rare ones — Dr (7), Rev (6), Col (2), and so on.

I'm going to group everything rare into a single "Rare" category, but I'm keeping
Master separate. Master is specifically a male child in British naming convention,
not just a rare title. The model needs to know that a "Master" is a child who
would have been prioritized in evacuation, not just a generic "other male."

The reason I'm grouping the others is a bias-variance tradeoff. If Rev appears
only 6 times, the model can't learn a stable pattern from 6 observations. It
might fit noise. A stable "Rare" group with enough examples is more reliable
than dozens of categories with 1-2 examples each.

In [ ]:
rare = [t for t in df['Title'].unique()
        if df['Title'].eq(t).sum() < 10 and t != 'Master']
df['Title'] = df['Title'].replace(rare, 'Rare')
print("Title distribution after grouping:")
print(df['Title'].value_counts())

In [ ]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone']    = (df['FamilySize'] == 1).astype(int)
df['HasCabin']   = df['Cabin'].notna().astype(int)

print(f"FamilySize: {df['FamilySize'].min()} to {df['FamilySize'].max()}")
print(f"Solo travelers: {df['IsAlone'].sum()} out of {len(df)}")

FamilySize goes up to 11. I looked that up — the Sage family, all 11 of them
traveling together, and not one survived. A tragic illustration of what large
family size meant in that situation.

Now let me check whether my hypothesis about medium-sized families doing better
is actually in the data.

In [ ]:
fam_surv = df.groupby('FamilySize')['Survived'].agg(['mean','count'])
fam_surv.columns = ['SurvivalRate', 'Count']
print(fam_surv)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(fam_surv.index, fam_surv['SurvivalRate'],
       color=GOLD, edgecolor='white', linewidth=0.5)
ax.set_xlabel('Family Size (including self)')
ax.set_ylabel('Survival Rate')
ax.set_title('Was I right about medium-sized families?')
ax.set_xticks(range(1, 12))
plt.tight_layout()
plt.savefig('family_survival.png', dpi=150, bbox_inches='tight', facecolor='#0A1628')
plt.show()

Partially right. Sizes 2, 3 and 4 do better than traveling alone. But sizes
5 and above drop sharply. So the pattern is: small family is better than solo,
but large family is the worst outcome.

This is non-linear, which matters for the model. I should keep FamilySize as a
numeric feature rather than just IsAlone, because the model needs to distinguish
between size 3 (good) and size 7 (bad). IsAlone alone would lose that information.

## Fixing the Age column

I established earlier that I can't drop the 177 rows with missing Age. The
straightforward approach would be to fill with the overall median (28 years).
But that ignores information I already have. A 1st-class female passenger in
1912 was statistically much older than a 3rd-class male passenger. I know their
class and sex. I should use that.

Let me verify that the group medians are actually different before I commit to this.

In [ ]:
group_medians = df.groupby(['Pclass','Sex'])['Age'].median()
print("Median age by class and sex:")
print(group_medians)
print(f"
Overall median: {df['Age'].median():.1f}")
print("
Difference from overall median:")
for (cls, sex), med in group_medians.items():
    diff = med - df['Age'].median()
    print(f"  Pclass={cls}, {sex}: {med:.1f} ({diff:+.1f} years)")

The differences are real. A 1st-class female median age is significantly higher
than a 3rd-class male. Using the single overall median would systematically
over-estimate ages for young 3rd-class passengers and under-estimate for older
1st-class passengers.

The grouped imputation is worth the extra code.

In [ ]:
age_medians = df.groupby(['Pclass','Sex'])['Age'].median()

def fill_age(row):
    if pd.isnull(row['Age']):
        return age_medians.loc[(row['Pclass'], row['Sex'])]
    return row['Age']

df['Age'] = df.apply(fill_age, axis=1)
print(f"Missing ages remaining: {df['Age'].isnull().sum()}")
print(f"Age now ranges from {df['Age'].min():.0f} to {df['Age'].max():.0f}")

## Deciding on the final features

Here is where I'm genuinely uncertain about something. I have SibSp, Parch,
and FamilySize. FamilySize is just SibSp + Parch + 1, so they're correlated.
Including all three in a Logistic Regression could cause multicollinearity —
where the model can't tell which one is doing the work and the coefficients
become unstable.

Let me check how correlated they actually are before deciding.

In [ ]:
num_cols = ['Pclass','Age','SibSp','Parch','Fare','FamilySize','IsAlone','HasCabin']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr.values, cmap='RdYlGn', vmin=-1, vmax=1)
ax.set_xticks(range(len(num_cols)))
ax.set_yticks(range(len(num_cols)))
ax.set_xticklabels(num_cols, rotation=45, ha='right')
ax.set_yticklabels(num_cols)
plt.colorbar(im, ax=ax)
ax.set_title('Correlation between numeric features')
for i in range(len(num_cols)):
    for j in range(len(num_cols)):
        ax.text(j, i, f'{corr.iloc[i,j]:.2f}',
                ha='center', va='center', color='black', fontsize=8)
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=150, bbox_inches='tight', facecolor='#0A1628')
plt.show()

FamilySize correlates at 0.89 with SibSp and 0.78 with Parch, as expected since
I constructed it from them. That is high.

My decision is to keep all three anyway, and here's my reasoning. I'm using
Logistic Regression with L2 regularization, which shrinks coefficients toward zero.
This is the standard tool for handling multicollinearity — the regularizer prevents
any one correlated feature from being assigned an arbitrarily large coefficient.

Also, FamilySize, SibSp and Parch capture conceptually different things.
SibSp is your spouse and siblings — your horizontal family. Parch is your parents
and children — your vertical family. FamilySize is the total unit. All three
could have different relationships with survival even if they're numerically correlated.

If the coefficients look unstable after fitting I'll revisit this decision.

## Building the preprocessing pipeline

I want all preprocessing to happen inside an sklearn Pipeline. The reason is
data leakage. If I compute the scaler or imputer on the full dataset before
splitting into folds, the validation folds technically "know" about the training
data through those computed statistics. The Pipeline ensures every preprocessing
step is fit only on the training fold and applied to the validation fold —
exactly as it would happen in production.

I learned this the hard way when my first cross-validation scores were
slightly too optimistic and I couldn't figure out why.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

NUM_FEATURES = ['Pclass','Age','SibSp','Parch','Fare','FamilySize','IsAlone','HasCabin']
CAT_FEATURES = ['Sex','Title','Embarked']
TARGET = 'Survived'

df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)

X = df[NUM_FEATURES + CAT_FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training: {X_train.shape[0]} rows, Test: {X_test.shape[0]} rows")
print(f"Survival rate — train: {y_train.mean():.3f}, test: {y_test.mean():.3f}")

In [ ]:
numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipe, NUM_FEATURES),
    ('cat', categorical_pipe, CAT_FEATURES)
])

model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier',   LogisticRegression(C=1.0, max_iter=1000, random_state=42))
])

print("Pipeline ready. Steps:", [s[0] for s in model.steps])

## Starting with a baseline — the dumbest possible model

I want to know what accuracy looks like if the model learns nothing.
If 62% of passengers didn't survive, a classifier that always says "didn't survive"
gets 62% accuracy for free. My logistic regression has to clearly beat that number
to be worth anything.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy_acc = cross_val_score(dummy, X_train, y_train, cv=cv, scoring='accuracy')

print("Dummy classifier (always predict majority class):")
print(f"  CV Accuracy: {dummy_acc.mean():.4f} ± {dummy_acc.std():.4f}")
print(f"  This is just the majority class rate: {y_train.value_counts(normalize=True).max():.4f}")

About 61-62% for free without learning anything. Now let's see what happens with
an actual model.

## Cross-validating the logistic regression

I'm using StratifiedKFold with 5 splits. Stratified means each fold has the same
survival ratio as the overall dataset — important because if one fold accidentally
got most of the survivors, the CV estimate would be unreliable.

I'll also report the standard deviation across folds. A high std would tell me the
model is sensitive to which data it sees, which would be a red flag for instability.

In [ ]:
cv_acc = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
cv_auc = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc')

print("Logistic Regression — 5-fold stratified CV:")
print(f"  Accuracy: {cv_acc.mean():.4f} ± {cv_acc.std():.4f}")
print(f"  Per fold: {[round(s,3) for s in cv_acc]}")
print()
print(f"  ROC-AUC:  {cv_auc.mean():.4f} ± {cv_auc.std():.4f}")
print(f"  Per fold: {[round(s,3) for s in cv_auc]}")
print()
print(f"  Improvement over dummy: +{cv_acc.mean() - dummy_acc.mean():.4f}")

About 82% accuracy and 0.868 ROC-AUC on the CV estimate. That's a significant
improvement over the 62% baseline.

But these are the cross-validation numbers — computed on the training data using
held-out folds. I need to be careful not to treat them as the final answer.
The honest number is the one I get on the hold-out test set that the model has
never seen in any form. I expect that number to be a little lower, because the
model was tuned on the training distribution.

This distinction matters. In my LinkedIn post I quoted 82% accuracy. That was
the CV number. The hold-out truth will be lower.

In [ ]:
model.fit(X_train, y_train)
y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:,1]

holdout_acc = accuracy_score(y_test, y_pred)
holdout_auc = roc_auc_score(y_test, y_proba)

print("Hold-out test set (never seen during training or CV):")
print(f"  Accuracy: {holdout_acc:.4f} ({holdout_acc*100:.2f}%)")
print(f"  ROC-AUC:  {holdout_auc:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=['Died','Survived']))

About 79% accuracy on the hold-out test. The CV estimate was 82%. That 3% gap
is the honest cost of generalizing to new data. It's small enough that I'm not
worried about overfitting, but it's real and I should report both numbers.

The ROC-AUC dropped from 0.868 to about 0.845. Same story.

Going forward: when I describe this model I will say "82% on cross-validation,
79% on hold-out" — not just one number. Reporting only the CV score would be
overstating the model's actual performance.

## What did the model actually learn?

The coefficients are the most important part for me. I want to know which features
the model weighted most, and whether those weights make sense given what I know
about the history.

One note before reading the numbers: the coefficients are only comparable to each
other because I standardized all numeric features with StandardScaler. Without that,
Fare (which ranges from 0 to 512) would have a smaller coefficient than Age just
because of the scale difference, not because it's less important.

In [ ]:
clf = model.named_steps['classifier']
pre = model.named_steps['preprocessor']

cat_names = (pre.named_transformers_['cat']
               .named_steps['onehot']
               .get_feature_names_out(CAT_FEATURES))

all_names = NUM_FEATURES + list(cat_names)
coefs = clf.coef_[0]

coef_df = pd.DataFrame({
    'Feature': all_names,
    'Coefficient': coefs,
    'AbsValue': np.abs(coefs)
}).sort_values('AbsValue', ascending=False)

print("Top 15 features by coefficient magnitude:")
print(coef_df.head(15)[['Feature','Coefficient']].to_string(index=False))

In [ ]:
top15 = coef_df.head(15)
colors = [GOLD if c > 0 else RED for c in top15['Coefficient']]

fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(top15['Feature'], top15['Coefficient'],
        color=colors, edgecolor='white', linewidth=0.3)
ax.axvline(x=0, color='white', linestyle='--', linewidth=0.8)
ax.set_xlabel('Coefficient (standardized — comparable across features)')
ax.set_title('What the model learned
Gold = increases survival probability, Red = decreases it')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('coefficients.png', dpi=150, bbox_inches='tight', facecolor='#0A1628')
plt.show()

Sex_female has the largest positive coefficient by a significant margin, around +2.6.
Converting to odds ratio: exp(2.6) is approximately 13.5. A female passenger had
about 13.5 times higher odds of surviving than a comparable male passenger, all
else being equal. That is a massive effect.

I knew sex would matter. I did not expect the magnitude to be that large.
The "women and children first" protocol wasn't just a guideline — it was enforced
so consistently that it produced a 13.5x odds ratio in the data. That's the kind
of number that makes you stop and think about what those hours on the ship must
have been like.

Title_Master came out with a positive coefficient, which is exactly right.
Master means male child, and children were prioritized. The title extraction
was worth doing — it captures something the raw Sex and Age columns don't
cleanly capture on their own.

Pclass has a negative coefficient. Remember that higher Pclass number means
lower class (3rd class = Pclass 3). So negative coefficient means being in
lower class decreases survival probability. Correct.

HasCabin has a meaningful positive coefficient. I created this as a simple proxy
for wealth — passengers with assigned cabin numbers were more likely to be
wealthier and housed on decks with better lifeboat access. The model is
using that signal.

One thing I'm still thinking about: Fare has a smaller coefficient than I expected
given that it's also a wealth proxy. My guess is that Pclass and HasCabin together
are absorbing most of the wealth signal, and Fare is doing less incremental work.
I could investigate this by running the model without Fare and comparing AUC,
but I'll leave that for a follow-up.

## Testing on passengers I know the historical answer for

This is not a rigorous evaluation — it's a sanity check. I want to give the
model three passenger profiles that I know the historical outcomes for, and see
whether its predictions align with the real story.

Thomas Andrews was the ship's designer. He knew better than anyone how badly
things were going. He stayed to help direct passengers to the lifeboats and died
when the ship went down. 1st class, male, middle-aged, no family aboard.

Molly Brown was a 1st class female passenger, American, wealthy. She helped
organize the evacuation and later became famous as "the unsinkable Molly Brown."
She survived.

The third profile is a generic 3rd-class young male traveling alone — the
demographic with historically the lowest survival rate.

In [ ]:
famous = pd.DataFrame({
    'Pclass':     [1,      1,     3],
    'Age':        [39,     44,    20],
    'SibSp':      [0,      0,     0],
    'Parch':      [0,      0,     0],
    'Fare':       [0,      27.7,  8.05],
    'FamilySize': [1,      1,     1],
    'IsAlone':    [1,      1,     1],
    'HasCabin':   [1,      1,     0],
    'Sex':        ['male','female','male'],
    'Title':      ['Mr',  'Mrs',  'Mr'],
    'Embarked':   ['S',   'C',    'S']
})

names = [
    'Thomas Andrews (1st class male, died)',
    'Molly Brown (1st class female, survived)',
    'Generic 3rd class young male'
]

probs = model.predict_proba(famous)[:, 1]

print("Survival probability predictions:")
for name, prob in zip(names, probs):
    bar = '█' * int(prob * 30)
    print(f"
{name}")
    print(f"  P(survive) = {prob:.3f}  |{bar:<30}|")

Thomas Andrews gets a low survival probability, which matches history.
Even though he was 1st class (which helps), being male overwhelms that advantage
in the model's coefficients. And historically that's exactly what happened —
his class position didn't save him.

Molly Brown gets a high probability. 1st class female is the highest-survival
demographic combination in the dataset.

The generic 3rd class young male gets a very low probability. Historically,
3rd class males had survival rates around 13-17%. The model is consistent
with that.

The predictions align with what I know. That gives me some confidence that
the model has captured real signal and not just memorized noise.

## Honest reflection on this analysis

The model gets about 82% cross-validation accuracy and 79% on held-out data.
It beats the "always predict the majority class" baseline by about 17-20
percentage points. ROC-AUC of 0.845 means it's ranking passengers in roughly
the right order of survival probability.

I also tested XGBoost. It got 83% on cross-validation, one percentage point
higher. I chose Logistic Regression over XGBoost for this notebook because
the coefficients tell a coherent story — sex, title, class, cabin access —
and I can verify that story against what I know about the historical event.
A black-box model that's 1% more accurate but opaque isn't more useful here.
The interpretability is the point.

There are three things I would do differently with more time.

The Age imputation is still a simplification. I used grouped medians, which
is better than a single median, but a proper approach would use a regression
model to predict Age from all other available features. I didn't do that
because it would make the pipeline more complex without dramatically improving
the results.

I didn't test dropping the correlated features (SibSp, Parch, FamilySize).
I argued that L2 regularization handles multicollinearity and kept all three.
I should have run the model both ways and compared to verify my reasoning.

Fare has a right-skewed distribution because a few passengers paid very large
amounts. A log transformation might help the model use that feature better.
I didn't try this.

What genuinely surprised me: the sex effect size. A 13.5 odds ratio is
extraordinary. I expected sex to matter. I didn't expect it to so completely
dominate every other variable. Whatever else this analysis shows, it is a
precise quantitative record of how the "women and children first" protocol
played out in practice.

---
James Koero — Junior ML Engineer — Kisumu, Kenya, May 2026